# Anomaly-Based Network Intrusion Detection System
## Notebook 05: Deep Learning Models

Implements an MLP classifier, an LSTM on sequence-shaped data,
and an Autoencoder for unsupervised anomaly detection.
All models use TensorFlow / Keras.

## 1. Imports

In [ ]:
import os, sys, warnings, joblib
warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import classification_report, roc_auc_score

from src.evaluation.metrics import evaluate_model, plot_training_history

tf.random.set_seed(42)
np.random.seed(42)

MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

print(f'TensorFlow {tf.__version__}')

## 2. Load Data

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
FEAT_DIR = os.path.join(PROJECT_ROOT, 'data', 'features')

X_train = pd.read_csv(f'{DATA_DIR}/X_train.csv')
X_val   = pd.read_csv(f'{DATA_DIR}/X_val.csv')
X_test  = pd.read_csv(f'{DATA_DIR}/X_test.csv')
y_train = pd.read_csv(f'{DATA_DIR}/y_train.csv').squeeze().values
y_val   = pd.read_csv(f'{DATA_DIR}/y_val.csv').squeeze().values
y_test  = pd.read_csv(f'{DATA_DIR}/y_test.csv').squeeze().values

SELECTED = joblib.load(f'{FEAT_DIR}/selected_features.pkl')

X_tr = X_train[SELECTED].values.astype('float32')
X_va = X_val[SELECTED].values.astype('float32')
X_te = X_test[SELECTED].values.astype('float32')

n_features = X_tr.shape[1]
print(f'Features: {n_features}  |  Train samples: {len(X_tr):,}')

## 3. MLP Classifier

In [ ]:
def build_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ], name='MLP')
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

mlp = build_mlp(n_features)
mlp.summary()

In [ ]:
es = callbacks.EarlyStopping(monitor='val_loss', patience=10,
                             restore_best_weights=True)
rl = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                  patience=5, min_lr=1e-6)

history_mlp = mlp.fit(
    X_tr, y_train,
    validation_data=(X_va, y_val),
    epochs=50, batch_size=256,
    callbacks=[es, rl], verbose=1
)

plot_training_history(history_mlp, title='MLP Training History')

In [ ]:
y_pred_mlp = (mlp.predict(X_te) > 0.5).astype(int).flatten()
y_prob_mlp = mlp.predict(X_te).flatten()

print('MLP Classification Report:')
print(classification_report(y_test, y_pred_mlp, target_names=['Normal', 'Attack']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_mlp):.4f}')

## 4. LSTM Classifier (sequence-shaped input)

In [ ]:
TIMESTEPS = 5

def make_sequences(X, timesteps):
    pad = np.zeros((timesteps - 1, X.shape[1]), dtype='float32')
    X_padded = np.vstack([pad, X])
    seqs = np.array([X_padded[i:i+timesteps] for i in range(len(X))],
                    dtype='float32')
    return seqs

X_tr_seq = make_sequences(X_tr, TIMESTEPS)
X_va_seq = make_sequences(X_va, TIMESTEPS)
X_te_seq = make_sequences(X_te, TIMESTEPS)

print(f'Sequence shape: {X_tr_seq.shape}')

In [ ]:
def build_lstm(timesteps, n_features):
    model = keras.Sequential([
        layers.Input(shape=(timesteps, n_features)),
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        layers.LSTM(64,  dropout=0.2, recurrent_dropout=0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1,  activation='sigmoid'),
    ], name='LSTM')
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

lstm_model = build_lstm(TIMESTEPS, n_features)
lstm_model.summary()

In [ ]:
history_lstm = lstm_model.fit(
    X_tr_seq, y_train,
    validation_data=(X_va_seq, y_val),
    epochs=30, batch_size=256,
    callbacks=[es, rl], verbose=1
)

y_pred_lstm = (lstm_model.predict(X_te_seq) > 0.5).astype(int).flatten()
y_prob_lstm = lstm_model.predict(X_te_seq).flatten()

print('\nLSTM Classification Report:')
print(classification_report(y_test, y_pred_lstm, target_names=['Normal', 'Attack']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_lstm):.4f}')

## 5. Autoencoder (Unsupervised Anomaly Detection)

In [ ]:
X_normal = X_tr[y_train == 0]   # train only on normal traffic

def build_autoencoder(input_dim):
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.Dense(32, activation='relu')(x)
    bottleneck = layers.Dense(16, activation='relu', name='bottleneck')(x)
    x = layers.Dense(32, activation='relu')(bottleneck)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='linear')(x)
    model = keras.Model(inputs, outputs, name='Autoencoder')
    model.compile(optimizer='adam', loss='mse')
    return model

ae = build_autoencoder(n_features)
ae.summary()

In [ ]:
history_ae = ae.fit(
    X_normal, X_normal,
    validation_split=0.1,
    epochs=50, batch_size=256,
    callbacks=[es], verbose=1
)

# Compute reconstruction error on test set
X_te_recon = ae.predict(X_te)
recon_error = np.mean(np.square(X_te - X_te_recon), axis=1)

# Choose threshold: 95th percentile of normal reconstruction errors
normal_recon = ae.predict(X_tr[y_train == 0])
normal_error = np.mean(np.square(X_tr[y_train == 0] - normal_recon), axis=1)
threshold = np.percentile(normal_error, 95)

y_pred_ae = (recon_error > threshold).astype(int)
print(f'Threshold (95th pct of normal errors): {threshold:.6f}')
print('\nAutoencoder Classification Report:')
print(classification_report(y_test, y_pred_ae, target_names=['Normal', 'Attack']))

## 6. Overall Comparison

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def quick_metrics(y_true, y_pred, y_prob=None):
    return {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan'),
    }

dl_results = {
    'MLP':         quick_metrics(y_test, y_pred_mlp,  y_prob_mlp),
    'LSTM':        quick_metrics(y_test, y_pred_lstm, y_prob_lstm),
    'Autoencoder': quick_metrics(y_test, y_pred_ae),
}

comparison_dl = pd.DataFrame(dl_results).T.round(4)
display(comparison_dl)

comparison_dl.plot(kind='bar', figsize=(11, 5), colormap='Set2')
plt.title('Deep Learning Model Comparison', fontsize=14)
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Save Deep Learning Models

In [ ]:
mlp.save(f'{MODELS_DIR}/mlp_model.keras')
lstm_model.save(f'{MODELS_DIR}/lstm_model.keras')
ae.save(f'{MODELS_DIR}/autoencoder_model.keras')
joblib.dump(float(threshold), f'{MODELS_DIR}/ae_threshold.pkl')

comparison_dl.to_csv(f'{RESULTS_DIR}/dl_model_comparison.csv')

print('All deep learning models saved.')
print('Pipeline complete!')